In [3]:
import jax
import equinox as eqx
import jax.random as jrandom
from diff_ml.reference_models.analytic import Analytic
from diff_ml.reference_models.bachelier import Bachelier
from diff_ml.reference_models.heston import Heston
from diff_ml.reference_models.mnist import MNIST_ref
from diff_ml.nn.utils import init_linear_weight, trunc_init
import optax
from diff_ml.nn.train import train

from diff_ml.utils import Range, MakeScalar, mse

from jax import vmap
import jax.numpy as jnp

from diff_ml.losses.directions import StreamingHessianSketch

In [4]:
def make_MNIST(key, test_set_order=2):
    ref_model = MNIST_ref(
        key=key, 
        scale=0.5,
        target_class=9
    )
    test_set = ref_model.get_test_set(n_samples=128, order=test_set_order)
    return ref_model, test_set
    

def make_bachelier(key, test_set_order=2):
    
    basket_dim = 7
    ref_model = Bachelier(
        key,
        basket_dim=basket_dim, 
        weights=jrandom.uniform(subkey, shape=(basket_dim,), minval=1.0, maxval=10.0)
    )
    test_set = ref_model.get_test_set(n_samples=8*1024, order=test_set_order)
    return ref_model, test_set

def make_heston(key, test_set_order=2):
    basket_dim = 7
    ref_model = Heston(
        key = key,
        basket_dim=basket_dim,
        basket_weights=jrandom.uniform(subkey, shape=(basket_dim,), minval=1.0, maxval=10.0)
    )
    if test_set_order ==3:
        test_set = ref_model.get_test_set(n_samples=128, order=test_set_order)
    else:
        test_set = ref_model.get_test_set(n_samples=2*1024, order=test_set_order)
    #y_std = test_set[4]["y_std"]
    #ref_model.set_y_std(y_std)
    return ref_model, test_set
    
def make_analytic(key, test_set_order=2):
    ref_model = Analytic(
        key=key,
        #type="RHE",
        ##type="Rastrigin",
        type="Rosenbrock",
        #type="Ackley",
        d=100,
        min_x=-1.5,
        max_x=2.5
    )
    #RHE: min_x = -5.0, max_x = 5.0
    ##Rastrigin: min_x = -5.12, max_x = 5.12
    #Rosenbrock: min_x = -1.5, max_x = 2.5
    #Ackley: min_x = -5.0, max_x = 5.0

    test_set = ref_model.get_test_set(n_samples=1*1024, order=test_set_order)
    return ref_model, test_set

In [8]:
from diff_ml.losses.regression import second_order_loss_fn
import time

#variant = "random"
variant = "batchSVD"
#variant = "fullHessian"

key = jrandom.PRNGKey(42)

key, subkey = jrandom.split(key)
ref_model, test_set = make_analytic(subkey, test_set_order=2)

batch_size = 256
k = 5


# nn model
key, subkey = jax.random.split(key)
input_dims = ref_model.n_dims
mlp = eqx.nn.MLP(key=subkey, in_size=input_dims, out_size="scalar", width_size=20, depth=3, activation=jax.nn.silu)
mlp = init_linear_weight(mlp, trunc_init, subkey)
surrogate_model = mlp
    


def process_batch(batch_key):
    batch = ref_model.sample(batch_key, batch_size, order=1)
    dirs_per_x = None
    Svals = None
    L2 = second_order_loss_fn(surrogate_model, batch, batch_key, ref_model, dirs_per_x, Svals, variant, k)
    return L2



n_batches = 100
batch_keys = jrandom.split(key, n_batches)

t0 = time.perf_counter()
losses = jax.vmap(process_batch)(batch_keys)
_ = jax.block_until_ready(losses)
t1 = time.perf_counter()
t =  t1 - t0
print(f"All batches took: {t:.4f} seconds")
    


All batches took: 2.6393 seconds


In [ ]:
main_key = jrandom.PRNGKey(42)


n_runs = 1

n_epochs = 100  # 200 for Heston single asset convergence
n_batches_per_epoch = 32#8#4#32#8#32 # 32
BATCH_SIZE = 256#16#8#256#16 #256 # 256

lr = 1e-3


k = 5
streaming_r = 8
test_set_order = 2
learnable_loss_weights = False
do_approx_metrics = False
do_test_eval = False



data = {}
#for variant in ["3rdBatchSVD"]:
for variant in ["value", "1st", "random", "pcady", "batchSVD", "perXSVD", "streaming", "fullHessian"]:#, "3rdBatchSVD"]:
    print("running variant", variant)
    
    y_errors = []
    dy_errors = []
    ddy_errors = []
    dddy_errors = []
    eFs = []
    e2s = []
    evrs = []
    engs = []
    avg_t_p_bs = []

    run_keys = jrandom.split(main_key, n_runs)
    for i, key in enumerate(run_keys):
        print("")
        print(f"run {i+1} / {n_runs} for {variant}")

        
        # reference model
        key, subkey = jax.random.split(key)

        #ref_model, test_set = make_bachelier(key=key, test_set_order=test_set_order)
        #ref_model, test_set = make_heston(key, test_set_order=test_set_order)
        ref_model, test_set = make_analytic(key, test_set_order=test_set_order)
        #ref_model, test_set = make_MNIST(key, test_set_order=test_set_order)
        

        
        
        # nn model
        key, subkey = jax.random.split(key)
        input_dims = ref_model.n_dims
        mlp = eqx.nn.MLP(key=subkey, in_size=input_dims, out_size="scalar", width_size=20, depth=3, activation=jax.nn.silu)
        mlp = init_linear_weight(mlp, trunc_init, subkey)
        surrogate_model = mlp
    
        # sketch
        sketch = None
        if variant == "streaming":
            key, subkey = jax.random.split(key)
            sketch = StreamingHessianSketch(
                        ref_model=ref_model,
                        r=streaming_r,
                        k=k, 
                        key=subkey)

        
    
        optim = optax.adam(learning_rate=lr)
    
    
        surrogate_model, iteration_datas, sketch, avg_time_per_batch = train(
                                model = surrogate_model, 
                                test_data=test_set,
                                optim=optim, 
                                n_epochs=n_epochs,
                                n_batches_per_epoch=n_batches_per_epoch,
                                batch_size=BATCH_SIZE,
                                ref_model=ref_model,
                                sketch=sketch,
                                variant=variant,
                                k=k,
                                learnable_loss_weights=learnable_loss_weights,
                                do_approx_metrics=do_approx_metrics,
                                do_test_eval=do_test_eval
                                )
        
    
        
        # eval price predictions
        test_pred_ys, test_pred_dys = vmap(jax.value_and_grad(surrogate_model))(test_set.x)
        test_pred_ddys = vmap(jax.hessian(MakeScalar(surrogate_model)))(test_set.x)
        test_pred_ddys = test_pred_ddys.reshape(test_set.ddy.shape)



        y_error = jnp.sqrt(mse(test_pred_ys, test_set.y))
        dy_error = jnp.sqrt(mse(test_pred_dys, test_set.dy))
        ddy_error = jnp.sqrt(mse(test_pred_ddys, test_set.ddy))

        if test_set_order > 2:
            test_pred_dddys = vmap(jax.jacfwd(jax.hessian(MakeScalar(surrogate_model))))(test_set.x)
            dddy_error = jnp.sqrt(mse(test_pred_dddys, test_set.dddy))        
        else:
            dddy_error = jnp.nan

        y_errors.append(y_error)
        dy_errors.append(dy_error)
        ddy_errors.append(ddy_error)
        dddy_errors.append(dddy_error)
        print(f"test y error: {y_error:.5f}")
        print(f"test dy error: {dy_error:.5f}")
        print(f"test ddy error: {ddy_error:.5f}")
        if test_set_order > 2:
            print(f"test dddy error: {dddy_error:.5f}")
    
        #print(iteration_datas[0])
        if "approximation metrics ref" in iteration_datas[0]:
            epochs = list(range(len(iteration_datas)))
            eF_vals  = jnp.array([iteration_datas[t]["approximation metrics ref"]["eF mean"]  for t in epochs])
            e2_vals  = jnp.array([iteration_datas[t]["approximation metrics ref"]["e2 mean"]  for t in epochs])
            evr_vals = jnp.array([iteration_datas[t]["approximation metrics ref"]["evr mean"] for t in epochs])
            eng_vals = jnp.array([iteration_datas[t]["approximation metrics ref"]["eng mean"] for t in epochs])
            mean_eF  = jnp.nanmean(eF_vals)
            mean_e2  = jnp.nanmean(e2_vals)
            mean_evr = jnp.nanmean(evr_vals)
            mean_eng = jnp.nanmean(eng_vals)
        else:
            mean_eF  = 0.0
            mean_e2  = 0.0
            mean_evr = 0.0
            mean_eng = 0.0
        eFs.append(mean_eF)
        e2s.append(mean_e2)
        evrs.append(mean_evr)
        engs.append(mean_eng)
        print(f"eF: {mean_eF:.5f}")
        print(f"e2: {mean_e2:.5f}")
        print(f"evr: {mean_evr:.5f}")
        print(f"eng: {mean_eng:.5f}")
        avg_t_p_bs.append(avg_time_per_batch)
        print(f"avg time per batch: {avg_time_per_batch:.5f}")

    # average the runs
    print("")
    y_errors = jnp.array(y_errors)
    dy_errors = jnp.array(dy_errors)
    ddy_errors = jnp.array(ddy_errors)
    dddy_errors = jnp.array(dddy_errors)

    y_errors_mean = y_errors.mean()
    dy_errors_mean = dy_errors.mean()
    ddy_errors_mean = ddy_errors.mean()
    dddy_errors_mean = dddy_errors.mean()
    y_errors_std = y_errors.std()
    dy_errors_std = dy_errors.std()
    ddy_errors_std = ddy_errors.std()
    dddy_errors_std = dddy_errors.std()

    print(f"y errors: \tmean: {y_errors_mean:.5f} \tstd: {y_errors_std:.5f}")
    print(f"dy errors: \tmean: {dy_errors_mean:.5f} \tstd: {dy_errors_std:.5f}")
    print(f"ddy errors: \tmean: {ddy_errors_mean:.5f} \tstd: {ddy_errors_std:.5f}")
    if test_set_order > 2:
        print(f"dddy errors: \tmean: {dddy_errors_mean:.5f} \tstd: {dddy_errors_std:.5f}")


    eFs = jnp.array(eFs)
    e2s = jnp.array(e2s)
    evrs = jnp.array(evrs)
    engs = jnp.array(engs)

    eFs_mean = eFs.mean()
    e2s_mean = e2s.mean()
    evrs_mean = evrs.mean()
    engs_mean = engs.mean()
    eFs_std = eFs.std()
    e2s_std = e2s.std()
    evrs_std = evrs.std()
    engs_std = engs.std()

    print(f"eFs: \tmean: {eFs_mean:.5f} \tstd: {eFs_std:.5f}")
    print(f"e2s: \tmean: {e2s_mean:.5f} \tstd: {e2s_std:.5f}")
    print(f"evrs: \tmean: {evrs_mean:.5f} \tstd: {evrs_std:.5f}")
    print(f"engs: \tmean: {engs_mean:.5f} \tstd: {engs_std:.5f}")

    avg_t_p_bs = jnp.array(avg_t_p_bs)


    print(f"avg time per batch: {avg_t_p_bs.mean():.5f}")



        


    data[variant] = {
        "y errors":   (y_errors_mean, y_errors_std),
        "dy errors":  (dy_errors_mean, dy_errors_std),
        "ddy errors": (ddy_errors_mean, ddy_errors_std),
        "dddy errors": (dddy_errors_mean, dddy_errors_std),
        "eFs":  (eFs_mean, eFs_std),
        "e2s":  (e2s_mean, e2s_std),
        "evrs": (evrs_mean, evrs_std),
        "engs": (engs_mean, engs_std),
        "avg time per batch": avg_t_p_bs.mean(),
    }

    

